In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

def main():
    print("1. Generating synthetic dataset...")
    # Generate a dataset with 1000 samples, 20 features, and a binary target
    X, y = make_classification(
        n_samples=1000, 
        n_features=20, 
        n_informative=15, 
        n_redundant=5, 
        random_state=42
    )
    
    print(f"Total dataset size: {X.shape[0]} rows\n")

    # ---------------------------------------------------------
    # 2. THE GOLDEN RULE: Split out the Holdout Test Set FIRST
    # ---------------------------------------------------------
    # We lock away 20% of the data. We will NOT touch this until the very end.
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, 
        test_size=0.20, 
        random_state=42, 
        stratify=y # Ensures the class balance is maintained in both splits
    )
    
    print(f"Development Set size (used for CV): {X_dev.shape[0]} rows")
    print(f"Holdout Test Set size (locked away): {X_test.shape[0]} rows\n")

    # ---------------------------------------------------------
    # 3. THE PIPELINE: Bundle Preprocessing and the Model
    # ---------------------------------------------------------
    # This guarantees that inside the Cross-Validation loop, the StandardScaler
    # only learns the mean/variance from the training folds, not the validation fold.
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(random_state=42))
    ])

    # ---------------------------------------------------------
    # 4. CROSS-VALIDATION SETUP
    # ---------------------------------------------------------
    # Using Stratified 5-Fold CV for our classification task
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # We will test different hyperparameter combinations during CV
    param_grid = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 5, 10]
    }

    print("5. Running Cross-Validation and Hyperparameter Tuning on Dev Set...")
    # GridSearchCV runs the CV loop for every combination of parameters above
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring='accuracy',
        n_jobs=-1, # Uses all available CPU cores
        verbose=1
    )

    # Fit the Grid Search ONLY on the Development Data
    grid_search.fit(X_dev, y_dev)

    print("\nCV Complete!")
    print(f"Best CV Accuracy found: {grid_search.best_score_:.4f}")
    print(f"Best Parameters: {grid_search.best_params_}\n")

    # ---------------------------------------------------------
    # 6. FINAL EVALUATION: Unlocking the Holdout Test Set
    # ---------------------------------------------------------
    print("6. Evaluating the final model on the unseen Holdout Test Set...")
    
    # grid_search automatically retrains the best model on the FULL X_dev dataset
    best_model = grid_search.best_estimator_
    
    # Predict on the test set that the model and scaler have NEVER seen
    y_pred = best_model.predict(X_test)
    
    # Generate the final performance metrics
    print("\n--- Final Classification Report ---")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()

1. Generating synthetic dataset...
Total dataset size: 1000 rows

Development Set size (used for CV): 800 rows
Holdout Test Set size (locked away): 200 rows

5. Running Cross-Validation and Hyperparameter Tuning on Dev Set...
Fitting 5 folds for each of 9 candidates, totalling 45 fits

CV Complete!
Best CV Accuracy found: 0.9113
Best Parameters: {'classifier__max_depth': None, 'classifier__n_estimators': 100}

6. Evaluating the final model on the unseen Holdout Test Set...

--- Final Classification Report ---
              precision    recall  f1-score   support

           0       0.94      0.91      0.92       100
           1       0.91      0.94      0.93       100

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.92       200
weighted avg       0.93      0.93      0.92       200

